In [ ]:
from pathlib import Path
import json
import os
import re
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 661
PROJECT_YEARS = list(range(2018, 2025))
PM25_THRESHOLD = 25.0
MIN_VALID_PM_HOURS = 18
MIN_VALID_COUNT_HOURS = 24
MAX_FIRE_DISTANCE_KM = 500.0
PREFILTER_DISTANCE_KM = 520.0
UPWIND_ANGLE_DEG = 45.0

# Transparent FSI research parameters. These are engineering defaults, not physical laws.
FSI_DISTANCE_TAU_KM = 100.0
FSI_AGE_HALF_LIFE_HOURS = 24.0
FSI_CONFIDENCE_WEIGHTS = {"l": 0.50, "n": 0.75, "h": 1.00}

# Optional future API acquisition. Historical Assessment 3 modelling continues to use
# the supplied 2018–2024 archive files so the evaluation period is reproducible.
USE_FIRMS_API = False
FIRMS_API_SOURCE = os.getenv("FIRMS_API_SOURCE", "VIIRS_SNPP_NRT")
FIRMS_API_BBOX = "125.5,-17.5,136.5,-7.5"  # west,south,east,north

possible_folders = [
    Path.cwd(),
    Path.cwd() / "Data Science",
    Path.home() / "Desktop" / "Data Science",
]

project_folder = None
for folder in possible_folders:
    if (folder / "ntepa_greater_darwin_hourly_2018_2024.csv").exists():
        project_folder = folder.resolve()
        break

if project_folder is None:
    raise FileNotFoundError(
        "Could not find ntepa_greater_darwin_hourly_2018_2024.csv. "
        "Place it and the seven FIRMS yearly archives (2018–2024) in the project folder."
    )

output_root = project_folder / "outputs_prt661_a3"
table_dir = output_root / "tables"
processed_dir = output_root / "processed"
for folder in [output_root, table_dir, processed_dir]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project folder:", project_folder)

In [ ]:
def fetch_firms_area_api(map_key, source=FIRMS_API_SOURCE, bbox=FIRMS_API_BBOX, day_range=1, date=None):
    """Fetch a recent FIRMS Area API CSV snapshot.

    This helper is intentionally optional. It is for future automated refreshes and is
    not mixed into the historical 2018–2024 evaluation dataset by default.
    """
    if not map_key:
        raise ValueError("A FIRMS MAP_KEY is required. Store it in the FIRMS_MAP_KEY environment variable.")
    if not 1 <= int(day_range) <= 5:
        raise ValueError("FIRMS Area API day_range must be between 1 and 5.")

    base = "https://firms.modaps.eosdis.nasa.gov/api/area/csv"
    url = f"{base}/{map_key}/{source}/{bbox}/{int(day_range)}"
    if date is not None:
        url += f"/{pd.Timestamp(date).strftime('%Y-%m-%d')}"
    return pd.read_csv(url)


if USE_FIRMS_API:
    api_key = os.getenv("FIRMS_MAP_KEY")
    api_snapshot = fetch_firms_area_api(api_key)
    api_snapshot.to_csv(processed_dir / "firms_api_latest_snapshot.csv", index=False)
    print("Saved optional FIRMS API snapshot:", len(api_snapshot), "rows")

In [ ]:
ntepa_file = project_folder / "ntepa_greater_darwin_hourly_2018_2024.csv"
df = pd.read_csv(ntepa_file)
df["datetime_local"] = pd.to_datetime(df["datetime_local"], errors="coerce")
df["date"] = df["datetime_local"].dt.normalize()
df["year"] = df["datetime_local"].dt.year

required_columns = {
    "datetime_local", "station", "latitude", "longitude",
    "pm25_ug_m3", "pm10_ug_m3", "relative_humidity_pct",
    "air_temperature_c", "wind_speed_m_s", "wind_direction_deg",
    "air_pressure_hpa", "rainfall_mm",
}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"NT EPA file is missing required columns: {sorted(missing_columns)}")

quality_columns = [
    "pm25_ug_m3", "pm10_ug_m3", "relative_humidity_pct",
    "air_temperature_c", "wind_speed_m_s", "wind_direction_deg",
    "air_pressure_hpa", "rainfall_mm",
]

overview = pd.DataFrame({
    "metric": [
        "rows", "columns", "date_min", "date_max", "stations",
        "full_row_duplicates", "station_timestamp_duplicates",
    ],
    "value": [
        len(df), df.shape[1], df["datetime_local"].min(), df["datetime_local"].max(),
        ", ".join(sorted(df["station"].dropna().astype(str).unique())),
        int(df.duplicated().sum()),
        int(df.duplicated(["station", "datetime_local"]).sum()),
    ],
})
overview.to_csv(table_dir / "a3_epa_raw_overview.csv", index=False)
print("\nRaw NT EPA overview:")
print(overview.to_string(index=False))

missing_by_year_station = (
    df.groupby(["year", "station"])[quality_columns]
      .agg(lambda s: s.isna().mean() * 100)
      .round(2)
      .reset_index()
)
missing_by_year_station.to_csv(table_dir / "a3_epa_missing_pct_year_station_variable.csv", index=False)

raw_summary = df[quality_columns].describe(
    percentiles=[0.001, 0.01, 0.05, 0.5, 0.95, 0.99, 0.999]
).T
raw_summary.to_csv(table_dir / "a3_epa_raw_numeric_summary.csv")

In [ ]:
df_clean = df.copy()
cleaning_log = []

# Remove exact duplicates and station-timestamp duplicates before aggregation.
before_rows = len(df_clean)
df_clean = df_clean.drop_duplicates().copy()
exact_removed = before_rows - len(df_clean)
cleaning_log.append({
    "variable": "row",
    "rule": "exact duplicate row removed",
    "values_set_or_changed": int(exact_removed),
})

before_rows = len(df_clean)
df_clean = (
    df_clean.sort_values(["station", "datetime_local"])
    .drop_duplicates(["station", "datetime_local"], keep="first")
    .copy()
)
station_time_removed = before_rows - len(df_clean)
cleaning_log.append({
    "variable": "row",
    "rule": "duplicate station-timestamp removed",
    "values_set_or_changed": int(station_time_removed),
})

def log_change(variable, rule, before_nonmissing, after_nonmissing):
    cleaning_log.append({
        "variable": variable,
        "rule": rule,
        "values_set_or_changed": int(before_nonmissing - after_nonmissing),
    })


for col in ["pm25_ug_m3", "pm10_ug_m3"]:
    before = int(df_clean[col].notna().sum())
    numeric = pd.to_numeric(df_clean[col], errors="coerce")
    df_clean.loc[numeric < 0, col] = np.nan
    after = int(df_clean[col].notna().sum())
    log_change(col, "negative concentration -> NaN", before, after)

# Station-level pressure audit and harmonisation.
pressure_profile = (
    df_clean.groupby("station")["air_pressure_hpa"]
    .agg(["count", "median", "min", "max"])
    .reset_index()
)
pressure_profile["conversion_factor"] = np.where(
    pressure_profile["median"].between(0.8, 1.2, inclusive="both"),
    1013.25,
    1.0,
)
pressure_profile.to_csv(table_dir / "a3_epa_pressure_unit_audit.csv", index=False)

for row in pressure_profile.itertuples(index=False):
    if row.conversion_factor != 1.0:
        mask = df_clean["station"].eq(row.station) & df_clean["air_pressure_hpa"].notna()
        df_clean.loc[mask, "air_pressure_hpa"] = (
            pd.to_numeric(df_clean.loc[mask, "air_pressure_hpa"], errors="coerce")
            * row.conversion_factor
        )
        cleaning_log.append({
            "variable": "air_pressure_hpa",
            "rule": f"{row.station}: atmosphere-scale pressure converted to hPa x1013.25",
            "values_set_or_changed": int(mask.sum()),
        })

plausible_ranges = {
    "relative_humidity_pct": (0.0, 100.0),
    "air_temperature_c": (-20.0, 60.0),
    "wind_speed_m_s": (0.0, 80.0),
    "air_pressure_hpa": (850.0, 1100.0),
}
for col, (low, high) in plausible_ranges.items():
    before = int(df_clean[col].notna().sum())
    numeric = pd.to_numeric(df_clean[col], errors="coerce")
    valid = numeric.between(low, high, inclusive="both")
    df_clean.loc[df_clean[col].notna() & ~valid, col] = np.nan
    after = int(df_clean[col].notna().sum())
    log_change(col, f"outside [{low}, {high}] -> NaN", before, after)

before = int(df_clean["wind_direction_deg"].notna().sum())
wind_numeric = pd.to_numeric(df_clean["wind_direction_deg"], errors="coerce")
valid_wind_dir = wind_numeric.between(0.0, 360.0, inclusive="left")
df_clean.loc[df_clean["wind_direction_deg"].notna() & ~valid_wind_dir, "wind_direction_deg"] = np.nan
after = int(df_clean["wind_direction_deg"].notna().sum())
log_change("wind_direction_deg", "outside [0, 360) -> NaN", before, after)

# Station-specific rainfall behaviour audit.
rain_profile_rows = []
for station, group in df_clean.sort_values("datetime_local").groupby("station"):
    s = pd.to_numeric(group["rainfall_mm"], errors="coerce")
    nonmissing = s.dropna()
    rain_profile_rows.append({
        "station": station,
        "nonmissing_pct": 100 * s.notna().mean(),
        "median_raw": nonmissing.median() if not nonmissing.empty else np.nan,
        "p95_raw": nonmissing.quantile(0.95) if not nonmissing.empty else np.nan,
        "zero_fraction_raw": nonmissing.eq(0).mean() if not nonmissing.empty else np.nan,
        "n_unique_raw": nonmissing.nunique(),
    })

rain_profile = pd.DataFrame(rain_profile_rows)
rain_profile["cumulative_counter_flag"] = (
    (rain_profile["median_raw"] > 100)
    & (rain_profile["zero_fraction_raw"].fillna(1.0) < 0.20)
)
rain_profile.to_csv(table_dir / "a3_epa_rainfall_station_behaviour_audit.csv", index=False)

df_clean = df_clean.sort_values(["station", "datetime_local"]).copy()
df_clean["rainfall_hourly_mm"] = np.nan
cumulative_stations = set(rain_profile.loc[rain_profile["cumulative_counter_flag"], "station"])

for station, idx in df_clean.groupby("station").groups.items():
    g = df_clean.loc[idx].sort_values("datetime_local")
    raw_rain = pd.to_numeric(g["rainfall_mm"], errors="coerce")
    if station in cumulative_stations:
        diffs = raw_rain.diff()
        hour_gap = g["datetime_local"].diff().dt.total_seconds().div(3600)
        derived = diffs.where(hour_gap.eq(1))
        derived = derived.where(derived >= 0)
        derived = derived.where(derived <= 300)
        df_clean.loc[g.index, "rainfall_hourly_mm"] = derived
    else:
        df_clean.loc[g.index, "rainfall_hourly_mm"] = raw_rain.where(raw_rain.between(0, 300))

cleaning_log_df = pd.DataFrame(cleaning_log)
cleaning_log_df.to_csv(table_dir / "a3_epa_cleaning_log.csv", index=False)

cleaned_ntepa_file = processed_dir / "NT_EPA_Air_Quality_Cleaned_2018_2024_A3.csv"
df_clean.to_csv(cleaned_ntepa_file, index=False)
print("\nCleaned NT EPA shape:", df_clean.shape)

In [ ]:
firms_files = sorted([
    p for p in project_folder.glob("fire_archive_SV-C2_*")
    if p.is_file() and p.suffix.lower() in {".csv", ".txt"}
])
if not firms_files:
    raise FileNotFoundError("No FIRMS yearly archives found matching fire_archive_SV-C2_* (.csv/.txt).")

print("\nFIRMS files found:", len(firms_files))
for file in firms_files:
    print(" -", file.name)

station_coordinates = {
    "Palmerston": (-12.507753, 130.948253),
    "Stokes Hill": (-12.459991, 130.847847),
    "Winnellie": (-12.424017, 130.893346),
}

def haversine_distance(lat, lon, station_lat, station_lon):
    earth_radius = 6371.0088
    lat1 = np.radians(np.asarray(lat, dtype=float))
    lon1 = np.radians(np.asarray(lon, dtype=float))
    lat2 = np.radians(station_lat)
    lon2 = np.radians(station_lon)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )
    a = np.clip(a, 0.0, 1.0)
    return 2 * earth_radius * np.arcsin(np.sqrt(a))


def initial_bearing_deg(source_lat, source_lon, target_lat, target_lon):
    """Initial bearing from source (fire) to target (monitoring station), 0–360 degrees."""
    lat1 = np.radians(np.asarray(source_lat, dtype=float))
    lon1 = np.radians(np.asarray(source_lon, dtype=float))
    lat2 = np.radians(float(target_lat))
    lon2 = np.radians(float(target_lon))
    dlon = lon2 - lon1
    y = np.sin(dlon) * np.cos(lat2)
    x = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(y, x)) + 360.0) % 360.0